# PatchCore ViT-B/16 Notebook (x224)

This curated notebook reviews the checked-in ViT PatchCore run from local artifacts.


## Submission Context

- Dataset notebook: `data/dataset/x224/benchmark_50k_5pct/notebook.ipynb`
- Dataset config: `data/dataset/x224/benchmark_50k_5pct/data_config.toml`
- Metadata CSV: `data/processed/x224/wm811k/metadata_50k_5pct.csv`
- Artifact root: `experiments/anomaly_detection/patchcore/vit_b16/x224/main/artifacts/patchcore_vit_b16_5pct/main_5pct`
- Mode: artifact-first review with checked-in checkpoint


## Imports and Saved Outputs

This cell loads the checked-in summary, evaluation files, checkpoint path, and saved UMAP outputs.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from tqdm import tqdm

cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]
REPO_ROOT = None
for candidate in candidate_roots:
    if (candidate / "src" / "wafer_defect").exists() and (candidate / "configs").exists():
        REPO_ROOT = candidate
        break
if REPO_ROOT is None:
    raise RuntimeError("Could not locate repo root containing src/wafer_defect and configs/")

SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from wafer_defect.evaluation import summarize_threshold_metrics

## Metrics and Plots

These cells surface the saved ViT evaluation metrics and regenerate the key plots into the standardized `plots/` folder.


In [ ]:
metrics_df = pd.DataFrame(
    [
        {"metric": "precision", "value": evaluation_metrics["anomaly_precision"]},
        {"metric": "recall", "value": evaluation_metrics["anomaly_recall"]},
        {"metric": "f1", "value": evaluation_metrics["anomaly_f1"]},
        {"metric": "roc_auc_z", "value": evaluation_metrics["roc_auc_z"]},
        {"metric": "avg_precision_z", "value": evaluation_metrics["avg_precision_z"]},
        {"metric": "threshold_raw", "value": summary["threshold_raw"]},
    ]
)
confusion_df = pd.DataFrame(evaluation_metrics["confusion_matrix"], index=["true_normal", "true_anomaly"], columns=["pred_normal", "pred_anomaly"])
display(metrics_df)
display(confusion_df)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(val_scores_df["score"], bins=30, alpha=0.85, color="#3d405b")
axes[0].axvline(threshold, color="red", linestyle="--")
axes[0].set_title("Validation Score Distribution")
axes[1].hist(test_scores_df[test_scores_df["is_anomaly"] == 0]["score"], bins=30, alpha=0.7, label="normal", color="#577590")
axes[1].hist(test_scores_df[test_scores_df["is_anomaly"] == 1]["score"], bins=30, alpha=0.7, label="anomaly", color="#e76f51")
axes[1].axvline(threshold, color="red", linestyle="--")
axes[1].set_title("Test Score Distribution")
axes[1].legend()
plt.tight_layout()
fig.savefig(PLOTS_DIR / "score_distribution.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(threshold_sweep_df["percentile"], threshold_sweep_df["precision"], label="precision")
ax.plot(threshold_sweep_df["percentile"], threshold_sweep_df["recall"], label="recall")
ax.plot(threshold_sweep_df["percentile"], threshold_sweep_df["f1"], label="f1")
ax.set_title("Threshold Sweep by Percentile")
ax.legend()
plt.tight_layout()
fig.savefig(PLOTS_DIR / "threshold_sweep.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)


## Defect and UMAP Review

This cell surfaces the saved defect breakdown and confirms the checked-in checkpoint and UMAP outputs.


In [ ]:
defect_breakdown_df = pd.read_csv(EVAL_DIR / "saved_defect_breakdown.csv")
display(defect_breakdown_df)

fig, ax = plt.subplots(figsize=(10, 5))
plot_df = defect_breakdown_df.sort_values("recall", ascending=False)
ax.barh(plot_df["failure_label"], plot_df["recall"], color="#81b29a")
ax.set_xlim(0.0, 1.0)
ax.invert_yaxis()
ax.set_title("Saved Defect Recall by Failure Label")
plt.tight_layout()
fig.savefig(PLOTS_DIR / "defect_breakdown.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

saved_outputs = {
    "checkpoint": str(CHECKPOINTS_DIR / "best_model.pt"),
    "umap_summary": str(UMAP_DIR / "umap_summary.json"),
    "umap_csv": str(UMAP_DIR / "umap_test_embeddings.csv"),
    "plots_dir": str(PLOTS_DIR),
}
saved_outputs
